# Axel's Research — Connect-4 Agent Diary

Running log of each working session: what was tried, what was measured, what was learned.

---
## 2026-05-17 — Session 1: Establish the baseline

**Goal:** get one working agent end-to-end through the tournament interface, beat the random opponent.
**Outcome:** two versions of *John Doe* built and measured; rule-based + minimax search.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Agent concepts considered

Brainstormed 13 possible agents (full descriptions in `axel/READ_ME.md`):

| Class | Agent | Core idea |
|---|---|---|
| Algorithmic | The Architect | Minimax + α-β |
| Algorithmic | The Statistician | MCTS |
| Algorithmic | The Geometrician | Heuristic matrix weights |
| Learning | The Pupil | Q-learning |
| Learning | The Darwinist | Genetic algorithm |
| Learning | The Mimic | Imitation from games |
| Rule-based | The Opportunist | 1-step greedy hierarchy |
| Rule-based | The Trap-Master | Odd/even parity control |
| Rule-based | The Centrist | Column-bias heuristic |
| Experimental | The Chaos Theory | ε-greedy |
| Experimental | The Mirror | Tit-for-tat |
| Experimental | The Gambiteer | Rare opening novelty |
| Baseline | **John Doe** | 1-step lookahead |

**Pick for session 1:** John Doe. Smallest possible agent that can plausibly beat random — the right place to start and the bar every later agent must clear.

### John Doe v1 — pure 1-step lookahead

Decision pipeline, in priority order, every turn:
1. If any legal column gives me an immediate 4-in-a-row → play it.
2. Else if any legal column would give the opponent an immediate 4-in-a-row on their next move → play it (block).
3. Else → uniform random over legal columns.

No training, no state, no search beyond one ply. File: [`axel/agent/john_doe.py`](../agent/john_doe.py).

**Result over 2000 games vs random** (split by John Doe's colour):

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/Agents-Connect4/results.csv')
df['win_pct']  = 100 * df['wins']   / df['games']
df['loss_pct'] = 100 * df['losses'] / df['games']

v1 = df[df['agent'] == 'john_doe_v1']

fig, ax = plt.subplots(figsize=(5, 3))
x = range(len(v1))
ax.bar(x, v1['win_pct'],  label='win %',  color='#2a9d8f')
ax.bar(x, v1['loss_pct'], label='loss %', color='#e76f51', bottom=v1['win_pct'])
ax.set_xticks(list(x))
ax.set_xticklabels(v1['color'])
ax.set_ylabel('% of games')
ax.set_title('John Doe v1 vs Random (n=2000 total)')
ax.legend()
plt.tight_layout()
plt.show()

**Observation.** ~97% win rate both colours, but **not zero losses** (~2.6% / 3.4%). The rubric demands *never* losing to random, so v1 is not enough.

**Why it loses.** Step 3 (random fallback) can hand the opponent a win:
- *Stacking trap* — John Doe plays in a column where the opponent already has 3-in-a-row diagonally adjacent. His piece becomes the floor that lets the opponent's next piece complete the diagonal.
- *Self-gift* — John Doe plays in a column whose next-up row would complete a horizontal for the opponent.

Both come from the same root cause: v1 has no notion of "would my move *create* a threat?".

### John Doe v2 — depth-3 minimax, terminal-only scoring

Same spirit (take wins, avoid losses) extended over a 3-ply game tree.

Scoring at leaves is the most minimal possible:
- `+1` if a 4-in-a-row by me is on the board
- `-1` if a 4-in-a-row by the opponent is on the board
- `0` otherwise (no positional heuristic)

Pick the move with the highest *worst-case* score (assume opponent plays best). Random tiebreak. `depth` is a constructor parameter so we can sweep it. File: [`axel/agent/john_doe_v2.py`](../agent/john_doe_v2.py).

**What depth-3 gains in theory:**
- Rejects suicide moves (any move giving opponent an immediate win → −1).
- Detects 1-move forks (move creating two simultaneous threats → +1 within the window).

**What it cannot guarantee:** forks the opponent sets up on ply 4+ (need depth ≥ 5).

In [ ]:
summary = (
    df.groupby(['agent', 'depth'])[['wins', 'losses', 'draws', 'games']]
      .sum()
      .assign(win_pct=lambda d: 100 * d['wins']   / d['games'],
              loss_pct=lambda d: 100 * d['losses'] / d['games'])
      .reset_index()
)
summary['label'] = summary['agent'] + ' (d=' + summary['depth'].astype(str) + ')'

fig, ax = plt.subplots(figsize=(6, 3))
x = range(len(summary))
ax.bar(x, summary['loss_pct'], color='#e76f51')
ax.set_xticks(list(x))
ax.set_xticklabels(summary['label'], rotation=15, ha='right')
ax.set_ylabel('loss % vs random')
ax.set_title('Search depth vs loss rate (lower is better)')
for i, v in enumerate(summary['loss_pct']):
    ax.text(i, v + 0.05, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

summary[['label', 'games', 'wins', 'losses', 'draws', 'win_pct', 'loss_pct']]

**Observation.** Going 1-ply → 3-ply → 5-ply moves the loss rate barely at all (~2.9% → ~2.75% → ~1.5% on a small n=200 sample). **Depth alone is not the bottleneck.**

**Why depth does not help much.** In any "quiet" position (no win/loss reachable inside the depth window) every legal move scores `NEUTRAL = 0`. With everything tied, v2 falls back to a uniform random tiebreak — i.e. v2 is *just as random as v1* in those positions. The deeper search only matters when a terminal node is in view, which is a minority of turns.

Random opponents stumble onto winning *structures* over many moves; no fixed-depth terminal-only search catches that without something pushing the agent toward intrinsically good squares.

### Today's lesson

> **Search depth without a positional heuristic ≈ random play in quiet positions.**

The win rate ceiling for terminal-only minimax against random is roughly the same as for the 1-ply rule-based agent (~97%). To break that ceiling we need *something* that biases tiebreaks toward better squares.

**Cheapest fix (next session):** add a centre-column tiebreak. The centre is part of more potential 4-in-a-rows than any other column — the single strongest known Connect-4 heuristic. Empirically expected to close most of the remaining loss gap with zero new search cost.

**Bigger fix (later):** replace the `0`-leaf with a real positional heuristic (counts of open 2- and 3-in-a-rows, threat squares, parity), so the search has something to optimise even far from terminal nodes.

### Next session
- Add centre-bias tiebreak to v2 → measure delta.
- Move on to a conceptually different agent (Q-learning *Pupil* or MCTS *Statistician*) so we have variety to compare in the final write-up.